# Live Results Figures — Krita Layout Transition Study

This notebook regenerates the **main Results figures** from the exported trial logs.

- Printed report figures are a snapshot at the time of writing.
- **Re-run all cells** after the CSV exports are updated to refresh the graphs on the latest complete participants.
- Incomplete participants listed in `meta.json` → `exclude` are omitted.

Open in Colab: use *Runtime → Run all*.


In [ ]:
# If opened in Google Colab, clone the repo (skip when already local).
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/RabiiAlaouiLamharzi/krita-final.git"

if IN_COLAB:
    ROOT = Path("/content/krita-final")
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
    DATA_DIR = ROOT / "docs" / "results_live"
else:
    DATA_DIR = Path(".").resolve()
    if not (DATA_DIR / "trials.csv").exists():
        for cand in [
            Path.cwd() / "docs" / "results_live",
            Path.cwd(),
            Path(__file__).resolve().parent if "__file__" in globals() else Path("."),
        ]:
            if (Path(cand) / "trials.csv").exists():
                DATA_DIR = Path(cand).resolve()
                break

print("DATA_DIR =", DATA_DIR)
assert (DATA_DIR / "trials.csv").exists(), "trials.csv not found — update exports or clone the repo"


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

MOVED = [
    "brush_tool", "line_tool", "rectangle_tool", "ellipse_tool",
    "move_tool", "gradient_tool", "fill_tool", "text_tool",
    "brush_preset", "eraser_preset",
    "add_layer", "delete_layer", "lower_layer", "raise_layer",
]
BLOCKS = ["S1-R1", "S1-R2", "S1-R3", "S2-R0", "S2-R1", "S2-R2", "S2-R3"]
COND_COLOR = {"Abrupt": "#0072B2", "Incremental": "#D55E00"}

meta = json.loads((DATA_DIR / "meta.json").read_text())
trials = pd.read_csv(DATA_DIR / "trials.csv")
surveys = pd.read_csv(DATA_DIR / "surveys.csv")

# bool columns may load as strings
for col in ("answered", "correct"):
    if col in trials.columns:
        trials[col] = trials[col].astype(str).str.lower().isin(["1", "true", "yes"])

moved = trials[trials.question_id.isin(MOVED)].copy()

print("Snapshot meta:", json.dumps(meta, indent=2))
print("Loaded trials:", len(trials), "| moved:", len(moved), "| participants:", trials.pid.nunique())
print("Unanswered:", int((~trials.answered).sum()),
      f"({100* (~trials.answered).mean():.1f}%)")


In [ ]:
def boot_ci_means(pp, measure, factors=("condition", "block"), n_boot=4000, seed=1):
    """Participant-level bootstrap CI for cell means."""
    rng = np.random.default_rng(seed)
    rows = []
    for keys, g in pp.groupby(list(factors)):
        if not isinstance(keys, tuple):
            keys = (keys,)
        vals = g[measure].to_numpy(dtype=float)
        if len(vals) == 0:
            continue
        boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
        row = {f: k for f, k in zip(factors, keys)}
        row.update({
            "mean": float(vals.mean()),
            "bci_low": float(np.quantile(boots, 0.025)),
            "bci_high": float(np.quantile(boots, 0.975)),
            "n": int(len(vals)),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def line_ci(ax, res, ylabel, ylim=None):
    for k, cond in enumerate(["Abrupt", "Incremental"]):
        sub = res[(res.condition == cond) & (res.block.isin(BLOCKS))].copy()
        sub["x"] = sub["block"].map({b: i for i, b in enumerate(BLOCKS)}) + (k - 0.5) * 0.1
        sub = sub.sort_values("x")
        ax.errorbar(
            sub["x"], sub["mean"],
            yerr=[sub["mean"] - sub["bci_low"], sub["bci_high"] - sub["mean"]],
            marker="o", markersize=4, capsize=3, linewidth=1.4,
            color=COND_COLOR[cond], label=cond,
        )
    ax.legend(frameon=False, fontsize=8)
    ax.set_ylabel(ylabel)
    if ylim:
        ax.set_ylim(*ylim)


## Exact location retrieval (14 relocated commands)

In [ ]:
exact_pp = (
    moved.groupby(["pid", "condition", "block"], as_index=False)
    .agg(exact=("correct", "mean"))
)
res_acc = boot_ci_means(exact_pp, "exact")

fig, ax = plt.subplots(figsize=(7.2, 3.4))
line_ci(ax, res_acc, "Mean exact retrieval", (0.35, 1.05))
ax.axvspan(2, 3, color="0.75", alpha=0.35, zorder=0, lw=0)
ax.annotate("48 h gap", xy=(2.5, 0.38), ha="center", fontsize=7, color="0.35")
ax.axvline(4.0, color="black", ls="--", lw=1, zorder=1)
ax.annotate("layout change begins", xy=(4.08, 0.38), fontsize=7)
ax.set_xticks(range(len(BLOCKS)))
ax.set_xticklabels(BLOCKS)
ax.set_xlabel("Recall test (S2-R0 = opening test on Layout A)")
fig.tight_layout()
plt.show()


## Switch cost and mean post-change retrieval

In [ ]:
wide = exact_pp.pivot_table(index=["pid", "condition"], columns="block", values="exact")
wide["switch"] = wide["S2-R1"] - wide["S2-R0"]
wide["post_all"] = wide[["S2-R1", "S2-R2", "S2-R3"]].mean(axis=1)
wide["post_no_last"] = wide[["S2-R1", "S2-R2"]].mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(8.0, 2.9))
specs = [
    (axes[0], "switch", "Change at switch\n(S2-R1 − S2-R0)", True),
    (axes[1], "post_no_last", "Mean after change\nwithout last recall\n(S2-R1, S2-R2)", False),
    (axes[2], "post_all", "Mean after change\nwith last recall\n(S2-R1–S2-R3)", False),
]
rng = np.random.default_rng(3)
for ax, col, title, hline in specs:
    for k, cond in enumerate(["Abrupt", "Incremental"]):
        vals = wide.xs(cond, level=1)[col].dropna().to_numpy()
        x = np.full(vals.size, k) + rng.uniform(-0.08, 0.08, vals.size)
        ax.scatter(x, vals, s=28, alpha=0.85, color=COND_COLOR[cond], zorder=3)
        ax.plot([k - 0.18, k + 0.18], [vals.mean()] * 2, color="black", lw=1.7)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Abrupt", "Incremental"], fontsize=7)
    ax.set_title(title, fontsize=8)
    if hline:
        ax.axhline(0, color="grey", lw=0.8, ls=":")
axes[0].set_ylabel("Exact retrieval")
fig.tight_layout()
plt.show()

# Bootstrap Abrupt − Incremental for post_all
rng = np.random.default_rng(1)
a = wide.xs("Abrupt", level=1)["post_all"].dropna().to_numpy()
b = wide.xs("Incremental", level=1)["post_all"].dropna().to_numpy()
diffs = np.array([rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean() for _ in range(10000)])
print(f"Mean post-change (all): Abrupt={a.mean():.2f}, Incremental={b.mean():.2f}")
print(f"Diff A−I = {a.mean()-b.mean():.2f}, 95% CI [{np.quantile(diffs,0.025):.2f}, {np.quantile(diffs,0.975):.2f}]")


## Serious-error rate (far wrong clicks / all trials)

In [ ]:
moved2 = moved.copy()
moved2["serious_trial"] = (
    moved2.answered & (~moved2.correct) & (moved2.slot_distance > 1)
).astype(float)
ser_pp = (
    moved2.groupby(["pid", "condition", "block"], as_index=False)
    .agg(serious_trial=("serious_trial", "mean"))
)
res_t = boot_ci_means(ser_pp, "serious_trial")

y_hi = max(0.35, float(res_t["bci_high"].max()) + 0.05)
fig, ax = plt.subplots(figsize=(7.2, 3.4))
line_ci(ax, res_t, "Serious-error rate\n(far clicks / all trials)", (0, y_hi))
ax.axvspan(2, 3, color="0.75", alpha=0.35, zorder=0, lw=0)
ax.annotate("48 h gap", xy=(2.5, 0.02), ha="center", fontsize=7, color="0.35")
ax.axvline(4.0, color="black", ls="--", lw=1, zorder=1)
ax.annotate("layout change begins", xy=(4.08, y_hi * 0.72), fontsize=7)
ax.set_xticks(range(len(BLOCKS)))
ax.set_xticklabels(BLOCKS)
ax.set_xlabel("Recall test")
fig.tight_layout()
plt.show()


## Difficulty ratings (post-recall)

In [ ]:
post = surveys[surveys.survey_type.astype(str).str.startswith("post_recall")].copy()
post["block"] = (
    "S" + post["sess"].astype(int).astype(str)
    + "-R" + post["survey_type"].astype(str).str.replace("post_recall_", "", regex=False)
)
post["difficulty"] = pd.to_numeric(post["survey_recall_difficulty_rating"], errors="coerce")
blocks6 = ["S1-R1", "S1-R2", "S1-R3", "S2-R1", "S2-R2", "S2-R3"]
colors = {1: "#2166ac", 2: "#67a9cf", 3: "#f7f7f7", 4: "#ef8a62", 5: "#b2182b"}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, cond in zip(axes, ["Abrupt", "Incremental"]):
    bottom_neg = np.zeros(len(blocks6))
    bottom_pos = np.zeros(len(blocks6))
    for score in [3, 2, 1, 4, 5]:
        shares = []
        for b in blocks6:
            vals = post[(post.condition == cond) & (post.block == b)]["difficulty"].dropna()
            shares.append((vals == score).mean() if len(vals) else 0.0)
        shares = np.array(shares)
        if score < 3:
            ax.barh(range(len(blocks6)), -shares, left=bottom_neg, color=colors[score], edgecolor="0.6", linewidth=0.3, label=str(score))
            bottom_neg = bottom_neg - shares
        elif score == 3:
            ax.barh(range(len(blocks6)), shares/2, left=-shares/2, color=colors[score], edgecolor="0.6", linewidth=0.3, label=str(score))
        else:
            ax.barh(range(len(blocks6)), shares, left=bottom_pos, color=colors[score], edgecolor="0.6", linewidth=0.3, label=str(score))
            bottom_pos = bottom_pos + shares
    ax.axvline(0, color="0.3", lw=0.8)
    ax.set_yticks(range(len(blocks6)))
    ax.set_yticklabels(blocks6)
    ax.set_title(cond)
    ax.set_xlim(-1, 1)
axes[0].set_xlabel("← easier | harder →")
axes[1].set_xlabel("← easier | harder →")
fig.suptitle("Post-recall difficulty (1=easy … 5=hard)", fontsize=10)
fig.tight_layout()
plt.show()


## How to update after new participants

1. Add new complete session folders under `experiment/data 4/` (or your current data folder).
2. Locally run: `python docs/results_live/refresh_export.py`
3. Commit and push the updated `trials.csv` / `surveys.csv` / `meta.json`.
4. Re-open this notebook (or *Runtime → Restart and run all* in Colab).
